In [1]:
from langchain_groq import ChatGroq

In [2]:
llm=ChatGroq(
    model_name="deepseek-r1-distill-llama-70b",
    temperature=0,
)
response = llm.invoke("What is the capital of France?")

In [3]:
response.content

'<think>\n\n</think>\n\nThe capital of France is Paris.'

In [4]:
from langchain.tools import tool

Custom tools

In [5]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers.
Args:
    a (int): The first number.
    b (int): The second number.
Returns:
    int: The product of a and b.
"""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds two numbers.
    Args:
        a (int): The first number.
        b (int): The second number.
    Returns:
        int: The sum of a and b.
    """
    return a + b

@tool
def divide(a: int, b: int) -> float:
    """Divides a by b.
    Args:
        a (int): The numerator.
        b (int): The denominator.
    Returns:
        float: The result of the division.
    """     
    if b == 0:
        raise ValueError("Division by zero is not allowed.")
    return a / b


In [6]:
from langchain_community.tools import DuckDuckGoSearchRun

In [7]:
search = DuckDuckGoSearchRun()

In [8]:
search.invoke("iPhone 17 release date")

c:\Swdtools\conda_envs\py311_agenticai\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


'Sep 9, 2025 · Apple today announced iPhone 17, featuring the new Center Stage front camera, a 48MP Fusion Main camera, and a new 48MP Fusion Ultra Wide camera. 6 days ago · iPhone Air, iPhone 17, iPhone 17 Pro Max sale start date, prices, availability in India: Apple has launched three new iPhone 17 models along with the ultra-slim iPhone Air, Apple … Apple will launch the iPhone 17 series in India on 9 September 2025, with prices expected to start at Rs. 79,990. Sep 9, 2025 · Tech News News: Apple has unveiled its latest iPhone 17 series, including the iPhone 17, Pro, Pro Max, and a new slim iPhone Air. All models boast ProMotion displays, Sep 10, 2025 · Apple held its annual iPhone event on Tuesday, September 9, to unveil the iPhone 17, ultra-thin iPhone Air, iPhone 17 Pro, and iPhone 17 Pro Max. ...'

In [9]:
tools = [multiply, add, divide, search]

In [10]:
llm_with_tools = llm.bind_tools(tools)

In [11]:
response = llm_with_tools.invoke("Hii")
response

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'Alright, the user sent "Hii". That\'s a friendly greeting, but not a specific question or task. I should respond in a way that\'s welcoming and offers assistance. Maybe ask how I can help them today. Keeping it open-ended encourages them to provide more details about what they need.\n'}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 388, 'total_tokens': 462, 'completion_time': 0.421409982, 'prompt_time': 0.043020145, 'queue_time': 0.105044125, 'total_time': 0.464430127}, 'model_name': 'deepseek-r1-distill-llama-70b', 'system_fingerprint': 'fp_1bbe7845ec', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--4c1d1a2d-70f8-4f25-bc21-b2a3fd2f9c04-0', usage_metadata={'input_tokens': 388, 'output_tokens': 74, 'total_tokens': 462})

In [12]:
response = llm_with_tools.invoke("what is 2+2? ")
response


AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking, "what is 2+2?" So, I need to figure out how to respond using the tools provided. Let me look at the available functions. There\'s add, multiply, divide, and duckduckgo_search. Since the question is about addition, the add function is the right choice.\n\nThe add function takes two integers, a and b, and returns their sum. So, I should call the add function with a=2 and b=2. That should give me 4. I need to format the tool call correctly in JSON within the XML tags as specified.\n\nI should make sure the function name is "add" and the arguments are a JSON object with a and b set to 2 each. That should do it.\n', 'tool_calls': [{'id': 'z0cv1wfnm', 'function': {'arguments': '{"a":2,"b":2}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 394, 'total_tokens': 583, 'completion_time': 0.856543717, 'prompt_time': 0.04745265, 'queue_time'

In [13]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END,MessagesState


In [14]:
builder = StateGraph(MessagesState)

In [15]:
user_query = ["tell me what is 2+2"]

In [16]:
SYSTEM_PROMPT= "you are a helpful assistant with using search and performing arithmetic on a set of inputs."
[SYSTEM_PROMPT]+ user_query

['you are a helpful assistant with using search and performing arithmetic on a set of inputs.',
 'tell me what is 2+2']

In [17]:
def function_1(state: MessagesState) -> MessagesState:
    user_question = state["messages"]
    input_question = [SYSTEM_PROMPT]+ user_question
    response = llm_with_tools.invoke(input_question)
    return {"messages":[response]}

In [18]:
builder.add_node("llm_decision_step", function_1)

In [19]:
from langgraph.prebuilt import ToolNode
builder.add_node("tools",ToolNode(tools))


In [20]:
builder.add_edge(START, "llm_decision_step")

In [21]:
from langgraph.prebuilt import tools_condition


builder.add_conditional_edges("llm_decision_step",
                              tools_condition)

In [22]:
builder.add_edge("tools", "llm_decision_step")

In [23]:
react_graph = builder.compile()

In [24]:
from IPython.display import Image, display
display(Image(react_graph.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 502.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:

message= [HumanMessage(content="what is 2 times the age of Elon Musk")]

In [ ]:
respose = react_graph.invoke({"messages":message})

c:\Swdtools\conda_envs\py311_agenticai\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


In [ ]:
response["messages"][-1].content